# [LAB-03] 3. 분석용 데이터 전처리 - 연습문제

## 준비작업

### 라이브러리 참조

In [1]:
from jussam import load_data
from helpers import *
from pandas import DataFrame, concat
from IPython.display import display, Markdown
import numpy as np

# 공간자기상관 점검을 위한 라이브러리 참조
# -> 패키지가 없다면 아래 한 줄의 주석을 해제하고 한 번만 실행한다.
# !pip install -q --upgrade geopandas libpysal pyproj esda
from geopandas import GeoDataFrame, points_from_xy
from libpysal.weights import Queen, Rook
from esda.moran import Moran


📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/
🔖 Version: 0.5.19


## 📚 겹치는 변수를 버릴 것인가, 나눠서 새로 짤 것인가

### 당신은 지난 두 단원에서 킹 카운티 주택 자료의 품질 점검과 탐색적 분석을 마친 분석가입니다.

1순위 `grade`, 2순위 `view`를 보고했더니 팀장이 되묻습니다. "규모 축 네 개가 서로 0.7 이상 겹쳐서 grade 하나로 대표했다고 했지. 그럼 나머지 셋은 그냥 버리는 거냐? 캘리포니아에서는 총량 변수들을 **서로 나눠서** 새 변수를 만들었잖아. 여기서도 해봐라."

말은 계속됩니다. "우편번호가 70개라 집단 비교가 어렵다며 통째로 뺐지. 지역 정보를 그렇게 버리는 게 맞나? 좌표는 그대로 있잖아. 그리고 상태 점수는 '유의하지만 효과가 없다'며 보류했는데, 보류는 결론이 아니다. 이번에 결론을 내라."

중복 행만 정리하고 값은 원본 그대로 둡니다. 종속변수는 주택 가격입니다. 만들 파생변수는 이렇게 정합니다.

**비율형 파생변수 5종** — 서로 나누면 "집이 크다"는 공통 성분이 약분되고 집의 **성격**만 남습니다.

| 파생변수 | 계산식 | 의미 |
|---|---|---|
| `sqft_per_bedroom` | `sqft_living / bedrooms` | 침실 하나당 거주 면적 |
| `living_ratio` | `sqft_living / sqft_lot` | 대지 대비 거주 면적 |
| `above_ratio` | `sqft_above / sqft_living` | 거주 면적 중 지상 비율 |
| `living_vs_neighbor` | `sqft_living / sqft_living15` | 인근 15채 대비 거주 면적 |
| `density_vs_neighbor` | `living_ratio / (sqft_living15 / sqft_lot15)` | 동네 평균 대비 밀도 |

마지막 `density_vs_neighbor`는 앞에서 만든 파생변수를 다시 재료로 쓰는 **2차 파생**입니다. 캘리포니아에서 가구 소득을 가구당 인구로 나눠 1인당 소득을 만들었던 것과 같은 방식입니다.

**범주형 파생변수 3종** — `is_renovated`는 리모델링 연도가 0보다 크면 1(1단원에서 "연도가 아니라 여부로 바꿔 써야 한다"고 결론낸 그 변수입니다), `has_basement`는 지하 면적이 0보다 크면 1, `condition_grp`는 주택 상태 점수를 4점 이상 `GOOD` / 3점 이하 `NORMAL` 두 집단으로 묶은 것입니다.

**공간 파생변수 1종** — 좌표를 바로 군집에 넣지 않습니다. 먼저 **Moran's I로 공간자기상관이 있는지 확인**하고, "가까운 집끼리 가격이 비슷하다"가 확인될 때에만 위도와 경도로 주택을 **5개 권역**으로 묶어 `spatial_cluster`를 만듭니다. 공간자기상관이 없다면 좌표를 파생변수로 바꿀 이유 자체가 없습니다.

> "파생변수는 만드는 게 끝이 아니다. 만든 다음에 **하나씩 검증해서** 살릴 것과 버릴 것을 갈라 와라."

검증이 끝나면 살아남은 변수만 모아 **파생변수 데이터셋**으로 저장하고, 여기에 로그 변환과 라벨링까지 적용한 **모델링용 최종 데이터셋**을 하나 더 남깁니다.

### 💻 코드 작성

#### 데이터 가져오기 (품질 점검 결과 반영)

In [2]:
origin = load_data("kc_house")

# 지난 단원들과 같은 출발점 : 중복 행만 정리하고 값은 원본 그대로 사용한다.
df = my_qtcheck.check_duplicates(origin)

target = "price"   # 종속변수 이름

print(f"데이터 크기: {df.shape}")
df.head()

📚 이 데이터 세트는 시애틀이 속한 킹 카운티의 주택 매매 가격 정보를 담고 있습니다. 2014년 5월부터 2015년 5월까지 판매된 주택들이 포함되어 있습니다.

(출처: https://www.kaggle.com/datasets/harlfoxem/housesalesprediction)



    field          description
--  -------------  ------------------------------------------------
 0  date           거래 날짜
 1  price          주택 가격 (USD 달러)
 2  bedrooms       침실 수 (개수)
 3  bathrooms      욕실 수 (개수)
 4  sqft_living    거주 공간 면적 (제곱 피트)
 5  sqft_lot       대지 면적 (제곱 피트)
 6  floors         층 수 (개수)
 7  waterfront     워터프론트 여부 (이진형: 1/0)
 8  view           조망 점수 (0~4)
 9  condition      주택 상태 점수 (1~5)
10  grade          건축 등급 점수 (1~13)
11  sqft_above     지상 면적 (제곱 피트)
12  sqft_basement  지하 면적 (제곱 피트)
13  yr_built       건축 연도
14  yr_renovated   리모델링 연도
15  zipcode        우편번호
16  lat            위도
17  long           경도
18  sqft_living15  15개 인근 주택의 평균 거주 공간 면적 (제곱 피트)
19  sqft_lot15     15개 인근 주택의 평균 대지 면적 (제곱 피트)

중복된 행의 수: 0
데이터 크기: (21613, 20)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.000,3,1.000,1180,5650,1.000,0,0,3,7,1180,0,1955,0,98178,47.511,-122.257,1340,5650
1,2014-12-09,538000.000,3,2.250,2570,7242,2.000,0,0,3,7,2170,400,1951,1991,98125,47.721,-122.319,1690,7639
2,2015-02-25,180000.000,2,1.000,770,10000,1.000,0,0,3,6,770,0,1933,0,98028,47.738,-122.233,2720,8062
3,2014-12-09,604000.000,4,3.000,1960,5000,1.000,0,0,5,7,1050,910,1965,0,98136,47.521,-122.393,1360,5000
4,2015-02-18,510000.000,3,2.000,1680,8080,1.000,0,0,3,8,1680,0,1987,0,98074,47.617,-122.045,1800,7503


#### 컬럼 의미를 정리한 딕셔너리

In [3]:
column_means = {
    "price":         "주택 가격 (USD)",
    "bedrooms":      "침실 수",
    "sqft_living":   "거주 공간 면적",
    "sqft_lot":      "대지 면적",
    "sqft_above":    "지상 면적",
    "sqft_basement": "지하 면적",
    "yr_renovated":  "리모델링 연도",
    "condition":     "주택 상태 점수",
    "sqft_living15": "인근 15채의 평균 거주 면적",
    "sqft_lot15":    "인근 15채의 평균 대지 면적"
}

#### 비율형 파생변수 생성

In [4]:
# 침실 하나당 거주 면적 : 방을 얼마나 넓게 쓰는 집인가
df["sqft_per_bedroom"] = df["sqft_living"] / df["bedrooms"]

# 대지 대비 거주 면적 : 땅을 얼마나 꽉 채워 지었나
df["living_ratio"] = df["sqft_living"] / df["sqft_lot"]

# 거주 면적 중 지상 비율 : 1에 가까울수록 지하실이 없는 집
df["above_ratio"] = df["sqft_above"] / df["sqft_living"]

# 인근 15채 대비 거주 면적 : 동네에서 큰 집인가 작은 집인가
df["living_vs_neighbor"] = df["sqft_living"] / df["sqft_living15"]

# 2차 파생 : 앞에서 만든 living_ratio 를 동네 평균 비율로 다시 나눈다
df["density_vs_neighbor"] = df["living_ratio"] / (df["sqft_living15"] / df["sqft_lot15"])

# 컬럼 의미 갱신
column_means.update({
    "sqft_per_bedroom":    "침실 하나당 거주 면적",
    "living_ratio":        "대지 대비 거주 면적",
    "above_ratio":         "거주 면적 중 지상 비율",
    "living_vs_neighbor":  "인근 15채 대비 거주 면적",
    "density_vs_neighbor": "동네 평균 대비 밀도"
})

derived_continuous = ["sqft_per_bedroom", "living_ratio", "above_ratio",
                      "living_vs_neighbor", "density_vs_neighbor"]

df[derived_continuous].head()

,sqft_per_bedroom,living_ratio,above_ratio,living_vs_neighbor,density_vs_neighbor
0,393.333,0.209,1.000,0.881,0.881
1,856.667,0.355,0.844,1.521,1.604
2,385.000,0.077,1.000,0.283,0.228
3,490.000,0.392,0.536,1.441,1.441
4,560.000,0.208,1.000,0.933,0.867


#### 생성한 파생변수 점검 (분모 · 결측 · 무한대)

In [5]:
# 분모로 사용한 변수의 최솟값 점검 (0이 있으면 inf 가 발생한다)
display(Markdown("### 분모로 사용한 변수의 최솟값"))
print(df[["bedrooms", "sqft_lot", "sqft_living", "sqft_living15", "sqft_lot15"]].min())

check = df[derived_continuous]

display(Markdown("### 파생 결과에 결측치가 생기지 않았는지 확인"))
print(check.isna().sum())

display(Markdown("### 파생 결과에 무한대가 생기지 않았는지 확인"))
inf_count = check.isin([np.inf, -np.inf]).sum()
print(inf_count)
print(f"\n무한대가 생긴 행: {int(inf_count.sum())}건")
print(f"침실 수가 0인 주택: {(df['bedrooms'] == 0).sum()}건")

### 분모로 사용한 변수의 최솟값

bedrooms           0
sqft_lot         520
sqft_living      290
sqft_living15    399
sqft_lot15       651
dtype: int64


### 파생 결과에 결측치가 생기지 않았는지 확인

sqft_per_bedroom       0
living_ratio           0
above_ratio            0
living_vs_neighbor     0
density_vs_neighbor    0
dtype: int64


### 파생 결과에 무한대가 생기지 않았는지 확인

sqft_per_bedroom       13
living_ratio            0
above_ratio             0
living_vs_neighbor      0
density_vs_neighbor     0
dtype: int64

무한대가 생긴 행: 13건
침실 수가 0인 주택: 13건


#### 무한대 처리 (전체의 5% 미만이면 해당 행 제거)

In [6]:
# 무한대가 전체의 몇 %인지부터 센다. 비율이 작으면 제거, 크면 대체가 원칙이다.
inf_mask = df[derived_continuous].isin([np.inf, -np.inf]).any(axis=1)
inf_ratio = inf_mask.mean()

print(f"무한대가 포함된 행: {int(inf_mask.sum())}건 / 전체 {len(df)}건 ({inf_ratio:.2%})")

if inf_ratio < 0.05:
    # 침실 0채는 1단원에서 "정상적인 값이 아니다"라고 기록해 둔 건이다.
    # 전체의 5%에 못 미치는 소수이므로 대체하지 않고 해당 행을 제거한다.
    df = df[~inf_mask].copy()
    print(f"→ 5% 미만이므로 해당 행을 제거했다. 남은 데이터 크기: {df.shape}")
else:
    # 5% 이상이면 제거할 경우 정보 손실이 크다.
    # 나눗셈이 성립하지 않은 자리는 결측으로 남겨 두고, 대체 여부는 뒤에서 판단한다.
    df[derived_continuous] = df[derived_continuous].replace([np.inf, -np.inf], np.nan)
    print("→ 5% 이상이므로 결측으로 변환했다. (대체 여부는 뒤에서 판단)")

display(Markdown("### 처리 후 무한대 · 결측 재점검"))
print(df[derived_continuous].isin([np.inf, -np.inf]).sum())
print(df[derived_continuous].isna().sum())


무한대가 포함된 행: 13건 / 전체 21613건 (0.06%)
→ 5% 미만이므로 해당 행을 제거했다. 남은 데이터 크기: (21600, 25)


### 처리 후 무한대 · 결측 재점검

sqft_per_bedroom       0
living_ratio           0
above_ratio            0
living_vs_neighbor     0
density_vs_neighbor    0
dtype: int64
sqft_per_bedroom       0
living_ratio           0
above_ratio            0
living_vs_neighbor     0
density_vs_neighbor    0
dtype: int64


#### 범주형 파생변수 생성

In [7]:
# 리모델링 여부 : 1단원에서 "연도가 아니라 여부로 바꿔 써야 한다"고 결론낸 변수
df["is_renovated"] = (df["yr_renovated"] > 0).astype(int)

# 지하실 유무 : 지하 면적의 0도 "없다"는 뜻이었다
df["has_basement"] = (df["sqft_basement"] > 0).astype(int)

# 주택 상태 점수 병합 : 4점 이상 GOOD / 3점 이하 NORMAL
df["condition_grp"] = df["condition"].apply(lambda x: "GOOD" if x >= 4 else "NORMAL")

column_means.update({
    "is_renovated":  "리모델링 여부",
    "has_basement":  "지하실 유무",
    "condition_grp": "주택 상태 구분"
})

for n in ["is_renovated", "has_basement", "condition_grp"]:
    freq = df[n].value_counts().rename("빈도").to_frame()
    freq["비율"] = df[n].value_counts(normalize=True)
    display(Markdown(f"#### ▶︎ {n}({column_means[n]}) 분포"))
    display(freq)

#### ▶︎ is_renovated(리모델링 여부) 분포

,빈도,비율
is_renovated,,
0,20686,0.958
1,914,0.042


#### ▶︎ has_basement(지하실 유무) 분포

,빈도,비율
has_basement,,
0,13113,0.607
1,8487,0.393


#### ▶︎ condition_grp(주택 상태 구분) 분포

,빈도,비율
condition_grp,,
NORMAL,14221,0.658
GOOD,7379,0.342


#### 공간자기상관 점검 (Moran's I)

In [8]:
# 위경도를 포인트 지오메트리로 만들고 거리 계산이 가능한 좌표계로 변환한다.
# -> EPSG:32610(UTM 10N)은 킹 카운티가 속한 워싱턴주 서부를 포함하는 투영 좌표계다.
geometry = points_from_xy(x=df["long"], y=df["lat"])

gdf = GeoDataFrame(df.drop(columns=["long", "lat"]), geometry=geometry, crs="EPSG:4326")
gdf = gdf.to_crs(epsg=32610)

# 인접 기반 공간가중치 행렬 생성 후 행 표준화("R" : 이웃 수 차이를 완화)
qw = Queen.from_dataframe(gdf, use_index=False)
qw.transform = "R"

rw = Rook.from_dataframe(gdf, use_index=False)
rw.transform = "R"

# 종속변수(price)에 공간자기상관이 있는지 확인한다.
moran_result = []

for w in [qw, rw]:
    m = Moran(gdf[target], w)
    moran_result.append({"Moran's I": m.I, "p-value": m.p_sim})

moran_df = DataFrame(moran_result, index=["Queen", "Rook"])
display(moran_df)

# 양수이면서 유의하면 "가까운 집끼리 가격이 비슷하다"는 뜻이므로,
# 좌표를 그대로 두지 않고 공간 파생변수로 바꿔 쓸 근거가 된다.
print("공간자기상관 유의 여부:", bool((moran_df["p-value"] < 0.05).all()))


,Moran's I,p-value
Queen,0.631,0.001
Rook,0.631,0.001


공간자기상관 유의 여부: True


#### 공간 파생변수 생성 (위경도 기반 군집)

In [9]:
# 우편번호 70개 대신, 좌표 두 개로 5개 권역을 만든다.
estimator, cdf, _ = my_cluster.kmeans(df, k=5, columns=["lat", "long"], plot=False)

df["spatial_cluster"] = cdf["그룹번호"].astype("category")
column_means["spatial_cluster"] = "위경도 기반 공간 군집"

# 권역별 규모 · 평균 가격 · 중심 좌표
display(df.groupby("spatial_cluster", observed=True).agg(
            건수=(target, "count"), 평균가격=(target, "mean"),
            위도=("lat", "mean"), 경도=("long", "mean")
        ).sort_values("평균가격", ascending=False))

StandardScaler 적용 (2개 컬럼)
컬럼                            변환 전                  변환 후
--------------------------------------------------------
lat                47.16 ~ 47.78          -2.92 ~ 1.57
long           -122.52 ~ -121.31          -2.17 ~ 6.39


,건수,평균가격,위도,경도
spatial_cluster,,,,
3,4698,753993.587,47.647,-122.132
0,6026,621950.459,47.686,-122.325
4,1857,578609.196,47.567,-121.933
2,4790,396115.334,47.476,-122.328
1,4229,332370.866,47.376,-122.141


#### 검증 대상 분류 및 타입 변환

In [10]:
derived_nominal = ["is_renovated", "has_basement", "condition_grp", "spatial_cluster"]

df = my_qtcheck.set_type(df, as_category=derived_nominal)

print("연속형 파생 :", derived_continuous)
print("범주형 파생 :", derived_nominal)

<class 'pandas.core.frame.DataFrame'>
Index: 21600 entries, 0 to 21612
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 21600 non-null  datetime64[ns]
 1   price                21600 non-null  float64       
 2   bedrooms             21600 non-null  int64         
 3   bathrooms            21600 non-null  float64       
 4   sqft_living          21600 non-null  int64         
 5   sqft_lot             21600 non-null  int64         
 6   floors               21600 non-null  float64       
 7   waterfront           21600 non-null  int64         
 8   view                 21600 non-null  int64         
 9   condition            21600 non-null  int64         
 10  grade                21600 non-null  int64         
 11  sqft_above           21600 non-null  int64         
 12  sqft_basement        21600 non-null  int64         
 13  yr_built             21600 non-null 

#### 단변량 분석 - 연속형 파생변수

In [11]:
derived_num_desc = my_qtcheck.numerical_summary(df, columns=derived_continuous)
derived_num_desc[['mean', '50%', 'skew', 'kurt', 'outliers_ratio', 'log_need']]

,mean,50%,skew,kurt,outliers_ratio,log_need
sqft_per_bedroom,618.153,576.667,1.570,5.792,0.032,log
living_ratio,0.324,0.248,2.376,10.446,0.062,log
above_ratio,0.875,1.000,-0.913,-0.734,0.000,none
living_vs_neighbor,1.053,1.000,2.039,11.957,0.069,log
density_vs_neighbor,1.074,1.000,21.942,1310.008,0.071,log


#### 이변량 분석 - 상관분석 (연속형 파생 → 종속변수)

In [12]:
corr_list = []

for f in derived_continuous:
    corr_list.append(my_stats.correlation(df, x=f, y=target, plot=False))

corr_result = concat(corr_list).sort_values("coef", ascending=False)
display(corr_result)

print("[ 상관 강도 판정별 변수 개수 ]")
print(corr_result['strength'].value_counts())

,,method,coef,p-value,strength,significant,normality_x,normality_y,linearity,influential_outlier,high_skew
x,y,,,,,,,,,,
sqft_per_bedroom,price,Spearman,0.558,0.000,Moderate,True,False,False,False,False,True
living_ratio,price,Spearman,0.280,0.000,Weak,True,False,False,False,True,True
living_vs_neighbor,price,Spearman,0.280,0.000,Weak,True,False,False,False,False,True
density_vs_neighbor,price,Spearman,0.172,0.000,Weak,True,False,False,False,False,True
above_ratio,price,Spearman,-0.187,0.000,Weak,True,False,False,False,False,True


[ 상관 강도 판정별 변수 개수 ]
strength
Weak        4
Moderate    1
Name: count, dtype: int64


#### 이변량 분석 - T검정 (2집단 범주형 파생 → 종속변수)

In [13]:
for n in derived_nominal:
    # 2개의 고유값만 존재하는 범주형 독립변수에 대해서만 검정을 수행한다.
    if len(df[n].unique()) == 2:
        display(Markdown(f"#### ▶︎ {n}({column_means[n]}) → {target}"))

        wide_df = my_prep.long2wide(df, hue=n, values=target)
        display(my_stats.test_independent(wide_df,
                    group1=wide_df.columns[0], group2=wide_df.columns[1], plot=False))

        display(df.groupby(n, observed=True)[target].agg(['count', 'mean']))

#### ▶︎ is_renovated(리모델링 여부) → price

statistic  p-value  significant result
test                alternative                                         
Mann-Whitney U test two-sided   6715277.000    0.000         True  0 ≠ 1
                    less        6715277.000    0.000         True  0 < 1
                    greater     6715277.000    1.000        False  0 ≤ 1

,count,mean
is_renovated,,
0,20686,530436.770
1,914,760379.030


#### ▶︎ has_basement(지하실 유무) → price

statistic  p-value  significant result
test                alternative                                          
Mann-Whitney U test two-sided   42169949.500    0.000         True  0 ≠ 1
                    less        42169949.500    0.000         True  0 < 1
                    greater     42169949.500    1.000        False  0 ≤ 1

,count,mean
has_basement,,
0,13113,486960.804
1,8487,622373.564


#### ▶︎ condition_grp(주택 상태 구분) → price

statistic  p-value  significant  \
test                alternative                                      
Mann-Whitney U test two-sided   52168596.000    0.490        False   
                    less        52168596.000    0.245        False   
                    greater     52168596.000    0.755        False   

                                        result  
test                alternative                 
Mann-Whitney U test two-sided    GOOD = NORMAL  
                    less         GOOD ≥ NORMAL  
                    greater      GOOD ≤ NORMAL

,count,mean
condition_grp,,
GOOD,7379,542279.474
NORMAL,14221,539070.475


#### 이변량 분석 - ANOVA (3집단 이상 범주형 파생 → 종속변수)

In [14]:
for n in derived_nominal:
    # 3개 이상의 고유값이 존재하는 명목형 독립변수에 대해서만 검정을 수행한다.
    if len(df[n].unique()) >= 3:
        display(Markdown(f"#### ▶︎ {n}({column_means[n]}) → {target}"))

        display(my_stats.anova_oneway(df, y=target, between=n))
        display(my_stats.posthoc_oneway(df, y=target, between=n, plot=False))

#### ▶︎ spatial_cluster(위경도 기반 공간 군집) → price

,test,Source,ddof1,ddof2,F,p_unc,np2,effect_size
0,welch_anova,spatial_cluster,4,8172.447,1743.023,0.000,0.185,Large


비교한 조합 쌍: 10개
유의한 쌍: 10개
평균 차이가 가장 큰 쌍: (1) vs (3) → 차이 421622.721, 효과크기 Large, p-value 0.000
평균 차이가 가장 작은 쌍: (0) vs (4) → 차이 43341.263, 효과크기 Negligible, p-value 0.000


,test,A,B,mean_A,mean_B,diff,se,T,df,pval,hedges,significant,effect_size
0,Games-Howell,0,1,621950.459,332370.866,289579.593,5276.038,54.886,7710.590,0.000,0.954,True,Large
1,Games-Howell,0,2,621950.459,396115.334,225835.125,6160.973,36.656,10533.252,0.000,0.679,True,Medium
2,Games-Howell,0,3,621950.459,753993.587,-132043.128,8312.668,-15.885,9075.412,0.000,-0.316,True,Small
3,Games-Howell,0,4,621950.459,578609.196,43341.263,7720.961,5.613,4602.612,0.000,0.122,True,Negligible
4,Games-Howell,1,2,332370.866,396115.334,-63744.468,4164.356,-15.307,7083.737,0.000,-0.311,True,Small
5,Games-Howell,1,3,332370.866,753993.587,-421622.721,6963.097,-60.551,5444.063,0.000,-1.226,True,Large
6,Games-Howell,1,4,332370.866,578609.196,-246238.330,6244.799,-39.431,2243.790,0.000,-1.406,True,Large
7,Games-Howell,2,3,396115.334,753993.587,-357878.253,7655.438,-46.748,7337.500,0.000,-0.965,True,Large
8,Games-Howell,2,4,396115.334,578609.196,-182493.862,7008.461,-26.039,3378.685,0.000,-0.712,True,Medium
9,Games-Howell,3,4,753993.587,578609.196,175384.391,8958.871,19.577,5838.099,0.000,0.426,True,Small


#### 다변량 분석 - 파생변수 간 중복 점검

In [15]:
corr = my_stats.multi_correlation(df, columns=derived_continuous, plot=False)
my_stats.correlation_summary(corr)

,sqft_per_bedroom,living_ratio,above_ratio,living_vs_neighbor,density_vs_neighbor
sqft_per_bedroom,1.000,0.216,-0.168,0.407,0.247
living_ratio,0.216,1.000,-0.153,0.305,0.474
above_ratio,-0.168,-0.153,1.000,-0.326,-0.246
living_vs_neighbor,0.407,0.305,-0.326,1.000,0.649
density_vs_neighbor,0.247,0.474,-0.246,0.649,1.000


,max-y,max-coef,count,columns
x,,,,
density_vs_neighbor,living_vs_neighbor,0.649,0,-
living_vs_neighbor,density_vs_neighbor,0.649,0,-
living_ratio,density_vs_neighbor,0.474,0,-
sqft_per_bedroom,living_vs_neighbor,0.407,0,-
above_ratio,living_vs_neighbor,-0.326,0,-


#### 모델링 투입 변수 채택 및 파생변수 데이터셋 저장

In [16]:
# 이번 단원의 검증 결과를 반영해 모형에 투입할 변수를 확정한다.
# -> condition_grp        : 병합 후에도 비유의(p=0.490) → 보류가 아니라 제외로 결론
# -> density_vs_neighbor  : 상관 최약(+0.172)에 왜도 최악(+21.9) → 제외
feature_cols = [
    "grade",                # 원본 유지 : 2단원 1순위
    "view",                 # 원본 유지 : 2단원 2순위
    "sqft_per_bedroom",     # 연속형 파생 : 이번 단원 2순위
    "living_ratio",         # 연속형 파생
    "above_ratio",          # 연속형 파생
    "living_vs_neighbor",   # 연속형 파생
    "is_renovated",         # 범주형 파생
    "has_basement",         # 범주형 파생
    "spatial_cluster"       # 공간 파생 : 이번 단원 1순위
]

print("투입 변수 :", len(feature_cols), "종")
print("투입 변수 목록 :", feature_cols)

# 저장 대상 = 투입 변수 + 종속변수
save_cols = feature_cols + [target]

# 로그 변환·라벨링 전, 원 단위 그대로의 상태를 먼저 남긴다.
df_derived = df[save_cols]
df_derived = my_qtcheck.set_type(df_derived,
                    as_category=["is_renovated", "has_basement", "spatial_cluster"])

df_derived.to_excel("kc_house_derived.xlsx", index=False)
print("저장 완료 :", df_derived.shape)


투입 변수 : 9 종
투입 변수 목록 : ['grade', 'view', 'sqft_per_bedroom', 'living_ratio', 'above_ratio', 'living_vs_neighbor', 'is_renovated', 'has_basement', 'spatial_cluster']
<class 'pandas.core.frame.DataFrame'>
Index: 21600 entries, 0 to 21612
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   grade               21600 non-null  int64   
 1   view                21600 non-null  int64   
 2   sqft_per_bedroom    21600 non-null  float64 
 3   living_ratio        21600 non-null  float64 
 4   above_ratio         21600 non-null  float64 
 5   living_vs_neighbor  21600 non-null  float64 
 6   is_renovated        21600 non-null  category
 7   has_basement        21600 non-null  category
 8   spatial_cluster     21600 non-null  category
 9   price               21600 non-null  float64 
dtypes: category(3), float64(5), int64(2)
memory usage: 1.4 MB


저장 완료 : (21600, 10)


#### 로그 변환 및 라벨링 후 최종 데이터셋 저장

In [17]:
# 투입 변수만 다시 기술통계량을 내어 로그 변환 대상을 확인한다.
feature_desc = my_qtcheck.numerical_summary(df_derived)
display(feature_desc[["skew", "kurt", "log_need"]])

# log_need 판정에 따라 변환 대상을 분류한다.
log_cols = feature_desc[feature_desc["log_need"] == "log"].index.tolist()
log1p_cols = feature_desc[feature_desc["log_need"] == "log1p"].index.tolist()
reflect_cols = feature_desc[feature_desc["log_need"] == "reverse_log1p"].index.tolist()

print("log(x)      :", log_cols)
print("log(1+x)    :", log1p_cols)
print("반사 후 log :", reflect_cols)

# 독립변수와 종속변수에 로그 변환 수행
df_log = my_prep.log_transform(df_derived,
            log_columns=log_cols,
            log1p_columns=log1p_cols,
            reflect_columns=reflect_cols)

# 범주형 라벨링 : category 타입을 모형이 읽을 수 있는 정수로 바꾼다.
label_cols = ["is_renovated", "has_basement", "spatial_cluster"]
df_label = my_prep.labeling(df_log, columns=label_cols)

df_label.to_excel("kc_house_features.xlsx", index=False)
print("저장 완료 :", df_label.shape)
df_label.head()


,skew,kurt,log_need
grade,0.782,1.148,log
view,3.397,10.898,log1p
sqft_per_bedroom,1.570,5.792,log
living_ratio,2.376,10.446,log
above_ratio,-0.913,-0.734,none
living_vs_neighbor,2.039,11.957,log
price,4.026,34.604,log


log(x)      : ['grade', 'sqft_per_bedroom', 'living_ratio', 'living_vs_neighbor', 'price']
log(1+x)    : ['view']
반사 후 log : []
컬럼        꼬리방향      변환식                   역변환식                                  왜도
----------------------------------------------------------------------------------------
grade     우측 꼬리     log(x)                exp(y)                    0.78 -> 0.19
sqft_per_bedroom우측 꼬리     log(x)                exp(y)                    1.57 -> 0.27
living_ratio우측 꼬리     log(x)                exp(y)                   2.38 -> -0.96
living_vs_neighbor우측 꼬리     log(x)                exp(y)                    2.04 -> 0.05
price     우측 꼬리     log(x)                exp(y)                    4.03 -> 0.43
view      우측 꼬리     log(1+x)              exp(y)-1                  3.40 -> 2.97
is_renovated (2종): 0=0, 1=1
has_basement (2종): 0=0, 1=1
spatial_cluster (5종): 0=0, 1=1, 2=2, 3=3, 4=4


저장 완료 : (21600, 10)


,grade,view,sqft_per_bedroom,living_ratio,above_ratio,living_vs_neighbor,is_renovated,has_basement,spatial_cluster,price
0,1.946,0.000,5.975,-1.566,1.000,-0.127,0,0,2,12.310
1,1.946,0.000,6.753,-1.036,0.844,0.419,1,1,0,13.196
2,1.792,0.000,5.953,-2.564,1.000,-1.262,0,0,0,12.101
3,1.946,0.000,6.194,-0.936,0.536,0.365,0,1,2,13.311
4,2.079,0.000,6.328,-1.571,1.000,-0.069,0,0,3,13.142


### 문제 풀이

#### 1. 다섯 종의 비율형 파생변수를 만든 뒤 점검했더니, 그중 한 변수에서만 나눗셈이 성립하지 않아 무한대가 나왔습니다. 무한대가 생긴 행은 몇 개인가요? (정수, 단위: 건)

- **정답**: `13`
- **복습 개념**: 파생변수 생성 직후의 점검입니다. 나눗셈으로 변수를 만들 때는 **분모에 0이 있는지**부터 확인해야 합니다. 0으로 나누면 결측이 아니라 무한대(`inf`)가 만들어지는데, 결측치 점검표는 빈 칸만 세기 때문에 이 값을 잡아내지 못합니다.
- **풀이 접근**: 분모로 쓴 변수들의 최솟값을 한 번에 확인해 0이 섞여 있는지 봅니다. 그다음 만들어진 다섯 열에 결측과 무한대가 각각 몇 개인지 세어 봅니다. 두 가지를 따로 세는 이유는 서로 다른 값이기 때문입니다. 마지막으로 무한대가 **전체의 몇 %인지**를 계산해 처리 방법을 정합니다.
- **근거(계산 결과)**: 분모 다섯 개 중 침실 수만 최솟값이 **0**이고, 나머지(대지 면적 520 / 거주 면적 290 / 인근 거주 면적 399 / 인근 대지 면적 651)는 모두 0보다 큽니다. 그래서 `sqft_per_bedroom` 한 열에만 무한대가 **13건** 생겼고, 이는 침실 수가 0인 주택 13채와 정확히 일치합니다. 결측치는 다섯 열 모두 0건입니다.
- **처리 방법**: 무한대가 포함된 행은 13건, 전체 21,613건의 **0.06%**로 5%에 한참 못 미칩니다. 이렇게 비율이 작을 때는 억지로 값을 채워 넣기보다 **해당 행을 제거**하는 편이 낫습니다. 없는 침실로 나눈 값은 애초에 의미가 없어서 평균이나 중앙값으로 메울 근거가 없기 때문입니다. 제거 후 데이터는 **21,600건**이 되고, 이후 분석은 모두 이 21,600건 위에서 이루어집니다. 반대로 비율이 5%를 넘었다면 제거는 표본 손실이 너무 커지므로 결측으로 바꿔 두고 대체 여부를 따로 판단해야 합니다.
- **함께 생각해 볼 점**: 이 13건은 1단원에서 값의 범위를 점검하며 "침실도 욕실도 없는 주택은 정상적인 값이 아니다"라고 기록만 해 두었던 바로 그 행들입니다. 그때는 고치지 않고 넘어갔지만, 파생변수를 만드는 순간 그 기록이 계산 오류로 되돌아왔습니다. **점검 단계에서 적어 둔 문제는 언젠가 반드시 청구됩니다.** 캘리포니아에서는 분모로 쓴 인구·가구 수에 0이 없어서 이 일이 일어나지 않았고, 같은 점검 코드가 "이상 없음"만 출력하고 끝났습니다.

#### 2. 다섯 종의 분포를 확인해 로그 변환 대상을 가려냈더니 넷은 대상이 되고 하나만 빠졌습니다. 대상에서 빠진 변수는 무엇인가요? (변수명 1개)

- **정답**: `above_ratio`
- **복습 개념**: 연속형 파생변수의 단변량 분석입니다. 왜도·첨도로 분포의 치우침을 읽고, 그 판정이 로그 변환 필요 여부로 이어집니다. 파생변수도 원본과 똑같이 이 점검을 거칩니다.
- **풀이 접근**: 다섯 개 파생변수만 골라 기술통계량 표를 만들고, 왜도 열과 로그 변환 필요 여부 열을 나란히 봅니다. 어느 쪽으로 얼마나 치우쳤는지가 판정의 근거입니다.
- **근거(계산 결과)**: **above_ratio**의 왜도는 **−0.913**로 유일하게 음수이고 판정도 `none`입니다. 나머지 넷은 모두 양의 왜도로 `log` 판정을 받았습니다(density_vs_neighbor +21.942 / living_ratio +2.376 / living_vs_neighbor +2.039 / sqft_per_bedroom +1.570).
- **헷갈리기 쉬운 점**: `above_ratio`가 변환 대상에서 빠진 것은 분포가 예쁘기 때문이 아닙니다. 이 값은 지상 면적을 거주 면적으로 나눈 것이라 **1을 넘을 수 없고**, 지하실이 없는 주택 13,113채가 전부 정확히 1에 몰려 있습니다. 위로 뻗을 꼬리가 아예 없으니 왜도가 음수로 나온 것이고, 로그 변환은 오른쪽 꼬리를 줄이는 도구라 여기서는 할 일이 없습니다. 반대로 `density_vs_neighbor`의 왜도 +21.942, 첨도 1,310.0은 2차 파생의 위험을 그대로 보여줍니다. 비율을 다시 비율로 나누면 **분모의 작은 값이 두 번 증폭**됩니다.

#### 3. 만든 다섯 종을 가격과 하나씩 견주었더니 상관 강도가 '보통'으로 판정된 것은 단 하나였습니다. 그 변수는 무엇인가요? (변수명 1개)

- **정답**: `sqft_per_bedroom`
- **복습 개념**: 연속형 파생변수의 이변량 분석입니다. 판정 기준은 원본 변수 때와 같습니다. 상관계수의 절댓값이 0.7 초과면 강함, 0.3 초과 0.7 이하면 보통, 0.3 이하면 약함입니다.
- **풀이 접근**: 파생변수 이름을 하나씩 돌면서 종속변수와의 상관분석을 수행하고 결과를 한 표로 합칩니다. 그 표의 강도 열을 값별로 세어 보면 '보통'이 몇 개인지 바로 나옵니다.
- **근거(계산 결과)**: **sqft_per_bedroom**이 ρ=**+0.558**로 `Moderate`이고, 나머지 넷은 모두 `Weak`입니다(living_vs_neighbor +0.280 / living_ratio +0.280 / density_vs_neighbor +0.172 / above_ratio −0.187). 다섯 종 모두 p=0.000으로 유의합니다.
- **실무 포인트**: 여기서 캘리포니아와 결과가 갈립니다. 그쪽에서는 파생변수 `income_per_person`(ρ=+0.722)이 원본 최강이던 `median_income`(+0.677)을 **넘어섰습니다.** 킹 카운티에서는 최강 파생인 `sqft_per_bedroom`(+0.558)이 2단원 1순위였던 `grade`(+0.658)에 미치지 못합니다. 이유는 집계 단위에 있습니다. 캘리포니아는 구역 평균이라 규모 성분이 신호를 덮고 있었고 나눗셈이 그 덮개를 걷어냈지만, 킹 카운티는 애초에 주택 한 채가 한 행이라 걷어낼 덮개가 별로 없었습니다. **같은 기법이 항상 같은 이득을 주지는 않습니다.** 다만 `above_ratio`의 부호가 음수라는 점은 기억해 두세요. 지상 비율이 낮을수록, 즉 지하실이 있을수록 비싸다는 뜻이고 총량 변수로는 볼 수 없던 관계입니다.

#### 4. 2단원에서 "유의하지만 효과크기가 없다"며 보류했던 주택 상태 점수를, 이번에는 두 집단으로 묶어 다시 검정했습니다. 이 검정의 유의확률은 얼마인가요? (소수 셋째 자리)

- **정답**: `0.490`
- **복습 개념**: 범주 재구성(병합)과 그 검증입니다. 표본이 적거나 효과가 미미한 범주를 묶어 집단을 단순하게 만든 뒤, **병합한 결과가 설명력을 유지하는지 다시 검정해서 확인**합니다.
- **풀이 접근**: 상태 점수를 4점 이상과 3점 이하 두 집단으로 묶고, 범주형 파생변수 중 고유값이 2개인 것만 골라 두 집단 비교를 수행합니다. 결과표의 유의확률 열과 집단별 평균을 함께 읽습니다.
- **근거(계산 결과)**: 양측검정 p = **0.490**로 0.05보다 훨씬 큽니다. 판정도 `False`이고 결과 열에는 `GOOD = NORMAL`이 찍힙니다. 집단 평균을 보면 GOOD 7,379채 542,279달러, NORMAL 14,221채 539,070달러로 차이가 **3,209달러**뿐입니다. 반면 같은 코드로 함께 검정된 리모델링 여부(760,379달러 대 530,437달러)와 지하실 유무(622,374달러 대 486,961달러)는 둘 다 p=0.000으로 유의합니다.
- **자주 하는 실수**: "범주를 병합하면 표본이 커지니 신호가 살아난다"고 기대하기 쉽습니다. 실제로는 반대 방향으로도 갑니다. 2단원에서 상태 점수는 p=0.000으로 유의했지만 효과크기 np2=0.007로 `Negligible`이었습니다. **그 유의성은 21,613건이라는 표본 크기가 만들어 준 것이었지 실제 차이가 아니었습니다.** 다섯 등급을 두 집단으로 묶자 그 미세한 차이마저 상쇄되어 유의성이 사라졌습니다. 캘리포니아의 `ocean_area`는 5범주를 2범주로 줄이고도 12만 달러의 집단 격차를 지켜냈으니 **병합 성공** 사례였고, 이쪽은 **병합해 봐도 살릴 것이 없었던** 사례입니다. 그래서 상태 점수는 보류가 아니라 제외로 결론을 냅니다.

#### 5. 이번 단원에서 새로 만든 변수 중 하나가 2단원 최고 효과크기였던 조망 점수(np2=0.168)마저 넘어섰습니다. 팀장에게 1순위로 보고할 이 변수는 무엇인가요? (변수명 1개)

- **정답**: `spatial_cluster`
- **복습 개념**: 공간자기상관 점검과 공간 파생변수 생성, 그리고 그 검증입니다. 좌표를 파생변수로 바꾸기 전에 **Moran's I**로 "가까운 집끼리 값이 비슷한가"를 먼저 확인합니다. 집단이 3개 이상인 범주형 변수이므로 집단 간 평균 비교로 검정하고, 채택 여부는 유의확률이 아니라 효과크기로 판단합니다.
- **풀이 접근**: 위경도로 공간 객체를 만들어 인접 기반 공간가중치 행렬(Queen · Rook)을 세우고 Moran's I를 구합니다. 공간자기상관이 확인되면 위도와 경도 두 열만 가지고 주택을 5개 권역으로 묶어 새 범주형 변수를 만듭니다. 범주형 파생 중 고유값이 3개 이상인 것만 골라 집단 간 평균 비교를 수행하고, 결과표의 효과크기 열을 2단원 결과와 비교합니다. 이어서 사후검정으로 어느 권역끼리 갈리는지 확인합니다.
- **근거(계산 결과)**: Moran's I는 Queen · Rook 모두 **0.631**(p=0.001)로 강한 양의 공간자기상관이 확인되어, 좌표를 권역으로 바꿀 근거가 섰습니다. 그렇게 만든 **spatial_cluster**의 Welch F=1,743.0(p=0.000), np2=**0.185**으로 판정은 `Large`입니다. 2단원의 조망 점수 0.168보다 큽니다. 권역별 평균 가격은 753,994달러(북동부)부터 332,371달러(남부)까지 벌어지고, 사후검정에서 10쌍이 모두 유의합니다. 최고·최저 권역의 격차는 421,623달러, hedges 1.226으로 `Large`입니다.
- **결론**: 팀장에게 드릴 답은 이렇게 정리됩니다. **1순위는 `spatial_cluster`, 2순위는 `sqft_per_bedroom`입니다.** 2단원에서 "지역이 70개라 집단 비교가 어렵다"며 통째로 버렸던 우편번호가, 좌표 두 개를 5개 권역으로 묶는 것만으로 데이터셋 최고의 설명변수가 되어 돌아왔습니다. **버릴 변수와 다시 만들 변수는 다릅니다.** 반대로 상태 점수는 병합해도 살아나지 않아 제외하고, `density_vs_neighbor`는 왜도 +21.942에 상관 +0.172로 분포도 설명력도 최약이라 우선 제거 후보입니다. 참고로 파생 5종끼리의 최대 상관은 `living_vs_neighbor`와 `density_vs_neighbor`의 0.649로, 원본 규모 축이 0.7~0.8대로 겹쳐 있던 것에 비하면 중복이 눈에 띄게 줄었습니다. **나눗셈이 상관계수를 올려 주지는 못했지만, 겹침을 풀어 주기는 했습니다.**
- **마무리(데이터셋 저장)**: 이 판단을 그대로 코드에 반영해 투입 변수를 **9종**(원본 `grade` · `view` + 연속형 파생 4종 + `is_renovated` · `has_basement` · `spatial_cluster`)으로 확정했습니다. 비유의 판정을 받은 `condition_grp`와 최약 판정을 받은 `density_vs_neighbor`는 여기서 빠집니다. 원 단위 그대로의 상태를 `kc_house_derived.xlsx`로 먼저 남기고, 로그 변환 대상(`grade` · `sqft_per_bedroom` · `living_ratio` · `living_vs_neighbor` · `price`는 `log`, `view`는 `log1p`, `above_ratio`는 변환 없음)과 라벨링까지 적용한 최종본을 `kc_house_features.xlsx`로 저장합니다. **변환 전 상태를 따로 남기는 이유는, 로그를 되돌려 원래 단위로 해석해야 할 일이 반드시 생기기 때문입니다.**
